In [ ]:
# MATLAB section 1/4 for PSTHEstimation: PSTH Estimation

# % PSTH Estimation
# We illustrate two ways to estimate a peristimulus time histogram using
# the nSTAT toolbox. One technique is the standard binning in time,
# averaging across trials, and dividing by the binwidth to estimate the
# spike rate and the other is based on the method presented in "Analysis of
# Between-Trial and Within-Trial Neural Spiking Dynamics" by Czanner et al
# in J Neurophysiology 2008.
#
#

# Python translation bootstrap for this helpfile.

# Topic: PSTHEstimation
# Execution group: full
# Workflow family: data
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/PSTHEstimation.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PSTHEstimation"
FAMILY = "data"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PSTHEstimation: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PSTHEstimation: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PSTHEstimation: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PSTHEstimation: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 3

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)


In [ ]:
# MATLAB section 2/4 for PSTHEstimation: Generate a known Conditional Intensity Function

# % Generate a known Conditional Intensity Function
# We generated a known conditional intensity function (rate function) and
# generate distinct realizations of point processes consistent with this
# rate function. We use the method of thinning to simulate a point process.
#
# MATLAB: close all;
# MATLAB: delta = 0.001;
# MATLAB: Tmax = 10;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: f=.2;
# MATLAB: lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0
# MATLAB: lambda = Covariate(time,lambdaData, '\Lambda(t)','time','s','Hz',{'\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB: numRealizations = 20; % Use 20 realization so that lamba and raster plot are the same size
# MATLAB: spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);
# MATLAB: spikeColl.plot; set(gca,'ytickLabel',[]);
# MATLAB: lambda.plot;
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: spikeColl.plot; set(gca,'ytickLabel',[]);
fig = FIGURE_TRACKER.new_figure(section_index=2, matlab_line_number=24, matlab_snippet="spikeColl.plot; set(gca,'ytickLabel',[]);")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PSTHEstimation.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 2, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/4 for PSTHEstimation: Estimate the PSTH with 500ms windows

# % Estimate the PSTH with 500ms windows
#
#
# MATLAB: figure;
# MATLAB: binsize = .5; %500ms window
# MATLAB: psth    = spikeColl.psth(binsize);
# MATLAB: psthGLM = spikeColl.psthGLM(binsize);
# MATLAB: trueRate = lambda; %rate*delta = expected number of arrivals per bin
# MATLAB: h1=trueRate.plot;
# MATLAB: h3=psthGLM.plot([],{{' ''k'',''Linewidth'',4'}});
# MATLAB: h2=psth.plot([],{{' ''rx'',''Linewidth'',4'}});
# MATLAB: legend off;
# MATLAB: legend([h1(1) h2(1)  h3(1)],'true','PSTH','PSTH_{glm}');
#
# Scalar summaries for automated parity checks.
# MATLAB: psth_mean_hz = mean(psth.data);
# MATLAB: psth_glm_mean_hz = mean(psthGLM.data);
# MATLAB: lambda_mean_hz = mean(lambda.data);
# MATLAB: parity = struct();
# MATLAB: parity.psth_mean_hz = psth_mean_hz;
# MATLAB: parity.psth_glm_mean_hz = psth_glm_mean_hz;
# MATLAB: parity.lambda_mean_hz = lambda_mean_hz;
#
# Because currently the psthGLM estimated the psth coefficients in each bin
# for each realization, we want the show the mean and standard error of the
# cofficient in each bin. We make the upper and lower confidence bounds
# equal to 1/sqrt(numRealization)=1/sqrt(psth.dimension) to view the
# standard error instead of the standard deviation
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=30, matlab_snippet="figure;")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PSTHEstimation_01.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 3, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/4 for PSTHEstimation: Section

# %
# Note the mean of the PSTH estimated via the GLM model and the PSTH
# computed via standard methods agree precisely. The benefit of the GLM
# estimated PSTH is the presence of confidence bounds on the estimate. Both
# the standard and GLM PSTH are in close agreement with the "true"
# underlying rate function (conditional intensity function) used in this
# simulated example. Both the PSTH and PSTHGLM code could be updated in the
# future to allow for variable bin sizes (e.g. in the vein of Baysian Adaptive Regression
# Splines by Wallstrom, Leibner and Kass). Alternatively, porting of BARS
# to Matlab may allow for it to be easily integrated into the nSTAT
# toolbox.
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: <synthetic MATLAB figure event #3 for PSTHEstimation>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=57, matlab_snippet="<synthetic MATLAB figure event #3 for PSTHEstimation>")
loaded_ref = FIGURE_TRACKER.save_reference_image(image_path="/private/tmp/upstream-nstat/helpfiles/PSTHEstimation_02.png")
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 4, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001;",
    "Tmax = 10;",
    "time = 0:delta:Tmax;",
    "f=.2;",
    "lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0",
    "lambda = Covariate(time,lambdaData, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "numRealizations = 20; % Use 20 realization so that lamba and raster plot are the same size",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "spikeColl.plot; set(gca,'ytickLabel',[]);",
    "lambda.plot;",
    "figure;",
    "binsize = .5; %500ms window",
    "psth    = spikeColl.psth(binsize);",
    "psthGLM = spikeColl.psthGLM(binsize);",
    "trueRate = lambda; %rate*delta = expected number of arrivals per bin",
    "h1=trueRate.plot;",
    "h3=psthGLM.plot([],{{' ''k'',''Linewidth'',4'}});",
    "h2=psth.plot([],{{' ''rx'',''Linewidth'',4'}});",
    "legend off;",
    "legend([h1(1) h2(1)  h3(1)],'true','PSTH','PSTH_{glm}');",
    "psth_mean_hz = mean(psth.data);",
    "psth_glm_mean_hz = mean(psthGLM.data);",
    "lambda_mean_hz = mean(lambda.data);",
    "parity = struct();",
    "parity.psth_mean_hz = psth_mean_hz;",
    "parity.psth_glm_mean_hz = psth_glm_mean_hz;",
    "parity.lambda_mean_hz = lambda_mean_hz;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PSTHEstimation.")

# Data-style workflow: trial-to-trial variability and PSTH-like estimates.
dt = 0.001
time = np.arange(0.0, 1.2, dt)
n_trials = 30

rate = 5.0 + 8.0 * (time > 0.35) + 4.0 * np.sin(2.0 * np.pi * 2.0 * time)
rate = np.clip(rate, 0.2, None)

trial_matrix = np.zeros((n_trials, time.size), dtype=float)
for k in range(n_trials):
    jitter = 0.6 + 0.8 * rng.random()
    p = np.clip(rate * jitter * dt, 0.0, 0.6)
    trial_matrix[k, :] = rng.binomial(1, p)

psth = trial_matrix.mean(axis=0) / dt
sem = trial_matrix.std(axis=0, ddof=1) / np.sqrt(n_trials) / dt

rates, prob_mat, sig_mat = DecodingAlgorithms.compute_spike_rate_cis(trial_matrix)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=False)
for k in range(min(18, n_trials)):
    t_spk = time[trial_matrix[k] > 0]
    axes[0].vlines(t_spk, k + 0.6, k + 1.4, linewidth=0.5)
axes[0].set_title(f"{TOPIC}: trial raster")
axes[0].set_ylabel("trial")

axes[1].plot(time, psth, color="tab:blue", linewidth=1.2)
axes[1].fill_between(time, psth - sem, psth + sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Hz")
axes[1].set_title("PSTH mean +/- SEM")

im = axes[2].imshow(prob_mat, aspect="auto", origin="lower", cmap="viridis")
axes[2].set_title("Trial-by-trial spike-rate p-values")
axes[2].set_xlabel("trial")
axes[2].set_ylabel("trial")
fig.colorbar(im, ax=axes[2], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("significant pair count", int(sig_mat.sum()))
assert np.allclose(prob_mat, prob_mat.T, atol=1e-12)
assert np.all(np.diag(prob_mat) == 1.0)

CHECKPOINT_METRICS = {
    "psth_mean_hz": float(np.mean(psth)),
    "significant_pairs": float(np.sum(sig_mat)),
}
CHECKPOINT_LIMITS = {
    "psth_mean_hz": (0.1, 50.0),
    "significant_pairs": (0.0, float(sig_mat.size)),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
